# Support Ticket Env — SFT Pre-Training
**OpenEnv x Scalar Hackathon**

Step 1 of 2-stage training: **Supervised Fine-Tuning** on gold-label examples.
This teaches the model correct JSON format + task logic before GRPO optimization.

- **Model:** Qwen/Qwen2.5-0.5B-Instruct
- **Algorithm:** SFT via `trl.SFTTrainer`
- **Dataset:** 1000 gold-label (prompt, completion) pairs
- **Runtime:** ~15-20 min on Kaggle T4
- **Output:** `/kaggle/working/sft-model` → used as base for GRPO

In [ ]:
!pip install -q 'trl>=0.18.2,<=0.24.0' 'transformers>=4.51.3,<=5.5.0' 'datasets>=3.4.1,<4.4.0' accelerate peft
!pip install -q requests matplotlib
print('Installation complete')

In [ ]:
import os, json, random

HF_TOKEN = ''
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except: pass
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or ''
    except: pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found.')

print('HF_TOKEN loaded OK')

MODEL_NAME  = 'Qwen/Qwen2.5-0.5B-Instruct'
RUNTIME     = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
SFT_OUT     = '/kaggle/working/sft-model'  if RUNTIME == 'kaggle' else '/content/sft-model'
RESULTS_IMG = '/kaggle/working/sft_results.png' if RUNTIME == 'kaggle' else '/content/sft_results.png'
HF_REPO_ID  = 'AlgoCore/support-ticket-sft-model'

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

import torch
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
print('Runtime:', RUNTIME, '| SFT output:', SFT_OUT)

In [ ]:
# ── Gold-label ticket bank (50 tickets, all categories) ──────────────────
ALL_TICKETS = [
    # billing (12)
    {'id':'B001','text':'I was charged twice for my subscription this month.','category':'billing','correct_action':'reply'},
    {'id':'B002','text':'My invoice shows a charge for a plan I never subscribed to.','category':'billing','correct_action':'escalate'},
    {'id':'B003','text':'I need an itemised invoice for my company accounting department.','category':'billing','correct_action':'reply'},
    {'id':'B004','text':'Why was I charged before my trial period ended?','category':'billing','correct_action':'reply'},
    {'id':'B005','text':'I switched plans but was still billed at the old rate.','category':'billing','correct_action':'reply'},
    {'id':'B006','text':'My payment method was charged three times in one day.','category':'billing','correct_action':'escalate'},
    {'id':'B007','text':'I cancelled my plan but the charge still appeared this month.','category':'billing','correct_action':'reply'},
    {'id':'B008','text':'Can you send me a receipt for my last payment?','category':'billing','correct_action':'reply'},
    {'id':'B009','text':'I was charged in USD but I signed up for GBP billing.','category':'billing','correct_action':'reply'},
    {'id':'B010','text':'The discount code I applied is not reflected in my invoice.','category':'billing','correct_action':'reply'},
    {'id':'B011','text':'I need to update my billing address on the invoice.','category':'billing','correct_action':'reply'},
    {'id':'B012','text':'My credit card was charged even though payment failed notification was sent.','category':'billing','correct_action':'escalate'},
    # account (10)
    {'id':'A001','text':'I cannot log into my account. Password reset email never arrives.','category':'account','correct_action':'reply'},
    {'id':'A002','text':'How do I cancel my subscription? I cannot find the option.','category':'account','correct_action':'reply'},
    {'id':'A003','text':'I want to change my email address associated with the account.','category':'account','correct_action':'reply'},
    {'id':'A004','text':'My account was locked after too many failed login attempts.','category':'account','correct_action':'reply'},
    {'id':'A005','text':'I accidentally deleted my account. Can it be restored?','category':'account','correct_action':'reply'},
    {'id':'A006','text':'I need to transfer my account to a different email.','category':'account','correct_action':'reply'},
    {'id':'A007','text':'Two-factor authentication is not working for my account.','category':'account','correct_action':'reply'},
    {'id':'A008','text':'I cannot find where to download my data for GDPR purposes.','category':'account','correct_action':'reply'},
    {'id':'A009','text':'My username was changed without my permission.','category':'account','correct_action':'escalate'},
    {'id':'A010','text':'I want to upgrade my account from free to premium.','category':'account','correct_action':'reply'},
    # technical (10)
    {'id':'T001','text':'Your app crashes every time I upload a file larger than 10 MB.','category':'technical','correct_action':'escalate'},
    {'id':'T002','text':'The API is returning 500 errors intermittently for 2 hours.','category':'technical','correct_action':'escalate'},
    {'id':'T003','text':'The dashboard is completely blank after the latest update.','category':'technical','correct_action':'escalate'},
    {'id':'T004','text':'Export to CSV is broken — it downloads an empty file.','category':'technical','correct_action':'escalate'},
    {'id':'T005','text':'Notifications are not being delivered to my email or phone.','category':'technical','correct_action':'escalate'},
    {'id':'T006','text':'The mobile app freezes on the login screen on iOS 17.','category':'technical','correct_action':'escalate'},
    {'id':'T007','text':'Search functionality returns no results for any query.','category':'technical','correct_action':'escalate'},
    {'id':'T008','text':'Data sync between devices stopped working 3 days ago.','category':'technical','correct_action':'escalate'},
    {'id':'T009','text':'The webhook integration keeps timing out and losing events.','category':'technical','correct_action':'escalate'},
    {'id':'T010','text':'Browser extension throws a JavaScript error on every page load.','category':'technical','correct_action':'escalate'},
    # refund (8)
    {'id':'R001','text':'I want a full refund. I have not used the service at all.','category':'refund','correct_action':'reply'},
    {'id':'R002','text':'I was double charged and need a refund for the extra payment.','category':'refund','correct_action':'reply'},
    {'id':'R003','text':'The product did not work as advertised. I want my money back.','category':'refund','correct_action':'reply'},
    {'id':'R004','text':'I cancelled within the 30-day window but have not received my refund.','category':'refund','correct_action':'reply'},
    {'id':'R005','text':'I would like a partial refund for the unused months of my annual plan.','category':'refund','correct_action':'reply'},
    {'id':'R006','text':'A refund was promised by your support agent 2 weeks ago but never arrived.','category':'refund','correct_action':'escalate'},
    {'id':'R007','text':'I need a refund processed urgently as it was a fraudulent charge.','category':'refund','correct_action':'escalate'},
    {'id':'R008','text':'How long does a refund take to appear on my credit card?','category':'refund','correct_action':'reply'},
    # general (10)
    {'id':'G001','text':'What are your business hours and do you have a phone number?','category':'general','correct_action':'reply'},
    {'id':'G002','text':'Thank you! The issue has been resolved. You guys are awesome.','category':'general','correct_action':'close'},
    {'id':'G003','text':'Do you offer a student discount or non-profit pricing?','category':'general','correct_action':'reply'},
    {'id':'G004','text':'Where can I find your terms of service and privacy policy?','category':'general','correct_action':'reply'},
    {'id':'G005','text':'Is your service available in my country? I am based in Brazil.','category':'general','correct_action':'reply'},
    {'id':'G006','text':'Can I use your product for commercial purposes?','category':'general','correct_action':'reply'},
    {'id':'G007','text':'Problem resolved, thanks for the quick response!','category':'general','correct_action':'close'},
    {'id':'G008','text':'Do you have an affiliate or referral program?','category':'general','correct_action':'reply'},
    {'id':'G009','text':'What integrations do you support with third-party tools?','category':'general','correct_action':'reply'},
    {'id':'G010','text':'I just wanted to say your product has been amazing for our team.','category':'general','correct_action':'close'},
]

# Gold reply templates per category
GOLD_REPLIES = {
    'billing':   'Thank you for reaching out about this billing issue. I can see the charge on your account. Our billing team will review your invoice and process any corrections. You will receive an updated receipt via email within 24 hours.',
    'account':   'Thank you for contacting us about your account. I understand how frustrating this can be. Please try resetting your password using the link sent to your registered email. If the issue persists, our account team will assist you directly.',
    'technical': 'I am escalating this technical issue to our engineering team immediately. They will investigate the root cause and provide a fix. You will receive a follow-up within 2 business hours.',
    'refund':    'I understand your refund request. Our refund policy allows for full refunds within 30 days. I am processing your refund now and it will appear on your credit card within 5-7 business days.',
    'general':   'Thank you for reaching out! Our support team is available Monday to Friday, 9am-6pm. You can also contact us via phone or email. We are happy to help with any questions you have.',
    'close':     'Thank you for letting us know the issue has been resolved. We are glad we could help! Please do not hesitate to contact us if you need anything else.',
}

SYSTEM_PROMPT = '''You are a customer support AI agent. Respond ONLY with a JSON object.

VALID action_type values: classify, reply, escalate, close
VALID category values: billing, technical, account, general, refund

For classify: {"action_type": "classify", "category": "<category>"}
For reply:    {"action_type": "reply", "reply_text": "<response>"}
For escalate: {"action_type": "escalate", "reply_text": "Escalating to engineering."}
For close:    {"action_type": "close", "reply_text": "Closing ticket."}

RULES:
- task_id=1: ALWAYS output action_type=classify first
- task_id=2: step=0 -> classify, step=1 -> reply/escalate/close
- task_id=3: step=0 -> classify, step=1 -> reply/escalate/close
- technical/crash/error/bug tickets -> escalate
- thank you/resolved tickets -> close
- billing/account/refund/general -> reply
- DO NOT use action_type=respond or action_type=resolve — those are INVALID'''

def make_prompt(ticket_text, task_id, current_category=None, feedback='New ticket.', step=0):
    user_msg = json.dumps({
        'ticket': ticket_text,
        'task_id': task_id,
        'current_category': current_category,
        'feedback': feedback,
        'step': step,
    })
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]

def gold_completion(ticket, task_id, step):
    """Generate the perfect gold-label completion for a given ticket + step."""
    cat    = ticket['category']
    action = ticket['correct_action']

    if task_id == 1:
        # Always classify
        return json.dumps({'action_type': 'classify', 'category': cat})

    elif task_id == 2:
        if step == 0:
            return json.dumps({'action_type': 'classify', 'category': cat})
        else:
            if action == 'escalate':
                return json.dumps({'action_type': 'escalate', 'reply_text': GOLD_REPLIES['technical']})
            elif action == 'close':
                return json.dumps({'action_type': 'close', 'reply_text': GOLD_REPLIES['close']})
            else:
                return json.dumps({'action_type': 'reply', 'reply_text': GOLD_REPLIES[cat]})

    else:  # task 3
        if step == 0:
            return json.dumps({'action_type': 'classify', 'category': cat})
        else:
            if action == 'escalate':
                return json.dumps({'action_type': 'escalate', 'reply_text': GOLD_REPLIES['technical']})
            elif action == 'close':
                return json.dumps({'action_type': 'close', 'reply_text': GOLD_REPLIES['close']})
            else:
                return json.dumps({'action_type': 'reply', 'reply_text': GOLD_REPLIES[cat]})

print(f'Ticket bank: {len(ALL_TICKETS)} tickets')
print('Sample gold completion (T001, task2, step0):', gold_completion(ALL_TICKETS[22], 2, 0))
print('Sample gold completion (T001, task2, step1):', gold_completion(ALL_TICKETS[22], 2, 1))

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'  # SFT uses right-padding

SEEDS    = list(range(200))
TASK_IDS = [1, 2, 3]

def build_sft_dataset():
    rows = []
    rng  = random.Random(42)
    for task_id in TASK_IDS:
        for seed in SEEDS:
            ticket = ALL_TICKETS[seed % len(ALL_TICKETS)]

            # Step 0: classify
            messages_0  = make_prompt(ticket['text'], task_id, None, 'New ticket. Classify it first.', 0)
            completion_0 = gold_completion(ticket, task_id, 0)
            # SFTTrainer expects 'text' column with full conversation
            full_0 = tokenizer.apply_chat_template(
                messages_0 + [{'role': 'assistant', 'content': completion_0}],
                tokenize=False
            )
            rows.append({'text': full_0, 'task_id': task_id, 'step': 0})

            # Step 1: resolve (tasks 2 & 3)
            if task_id in (2, 3):
                messages_1   = make_prompt(
                    ticket['text'], task_id,
                    ticket['category'],
                    f"Category set to {ticket['category']}. Now resolve the ticket.",
                    1
                )
                completion_1 = gold_completion(ticket, task_id, 1)
                full_1 = tokenizer.apply_chat_template(
                    messages_1 + [{'role': 'assistant', 'content': completion_1}],
                    tokenize=False
                )
                rows.append({'text': full_1, 'task_id': task_id, 'step': 1})

    rng.shuffle(rows)
    # Split 90/10 train/val
    split = int(len(rows) * 0.9)
    return Dataset.from_list(rows[:split]), Dataset.from_list(rows[split:])

train_ds, val_ds = build_sft_dataset()
print(f'SFT dataset — train: {len(train_ds)}, val: {len(val_ds)}')
print('Sample text (first 400 chars):')
print(train_ds[0]['text'][:400])

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

print(f'Loading {MODEL_NAME}...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map={'': 0},
    token=HF_TOKEN,
)
model.config.use_cache = False

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
)

sft_config = SFTConfig(
    output_dir=SFT_OUT,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='no',
    report_to='none',
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
    max_seq_length=512,
    dataset_text_field='text',
)

print(f'SFTTrainer ready | train={len(train_ds)} val={len(val_ds)}')
print('Starting SFT...')
print('=' * 60)

sft_result = trainer.train()

print('=' * 60)
print(f'SFT complete! Loss: {sft_result.training_loss:.4f}')

In [ ]:
import os
os.makedirs(SFT_OUT, exist_ok=True)
trainer.save_model(SFT_OUT)
tokenizer.save_pretrained(SFT_OUT)
print(f'SFT model saved to {SFT_OUT}')

# Push to HF Hub
try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO_ID, exist_ok=True, private=False)
    api.upload_folder(folder_path=SFT_OUT, repo_id=HF_REPO_ID, repo_type='model')
    print(f'SFT model pushed to: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'HF push failed: {e}')

print('\n✅ SFT done. Now run train_grpo.ipynb and set MODEL_NAME to:')
print(f'   MODEL_NAME = "{SFT_OUT}"  # or "{HF_REPO_ID}"')

In [ ]:
# Quick eval to verify SFT worked
import re

model.eval()
model.config.use_cache = True

def sft_generate(ticket_text, task_id, step=0, current_category=None):
    messages = make_prompt(ticket_text, task_id, current_category, 'New ticket.', step)
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=80, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, use_cache=True
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== SFT Quick Eval ===')
test_cases = [
    ('I was charged twice this month.', 1, 0, None),
    ('App crashes on file upload.', 1, 0, None),
    ('Thank you, issue resolved!', 1, 0, None),
    ('I want a refund.', 2, 0, None),
    ('I want a refund.', 2, 1, 'refund'),
    ('API returning 500 errors.', 3, 0, None),
    ('API returning 500 errors.', 3, 1, 'technical'),
]
for text, tid, step, cat in test_cases:
    out = sft_generate(text, tid, step, cat)
    print(f'T{tid} step{step}: [{text[:35]}] -> {out[:80]}')